# Using and Building a Prediction Market

## Introduction

#### The Core Concept: Information Aggregation
*   **Definition:** Marketplaces where participants trade shares of future event outcomes. The price of a share directly reflects the market's perceived probability of that outcome.
*   **How it Works:** Financial incentives drive accuracy. If a trader believes the current price is wrong, they buy underpriced shares or sell overpriced ones to profit. This continuous trading automatically aggregates disparate private knowledge into a single, accurate collective forecast.

#### Eliciting Truth: From Scoring Rules to Markets
*   **Proper Scoring Rules (Traditional):** Mathematical formulas (e.g., Brier score, Log score) designed to reward forecasters based on accuracy. They ensure that a forecaster's most profitable strategy is to report their true belief. 
    *   *Limitation:* They are "one-shot" mechanisms, making it difficult to aggregate multiple people's opinions simultaneously.
*   **Market Scoring Rules:** Introduced by Robin Hanson, this mechanism transitions scoring to a sequential market setting. Each trader effectively corrects the previous trader's prediction, updating the probability and establishing a new consensus.

#### The Engine: LMSR and Automated Market Makers (AMMs)
*   **LMSR (Logarithmic Market Scoring Rule):** The most popular mathematical implementation of a market scoring rule. It functions as an **Automated Market Maker (AMM)** that is always willing to take the opposite side of any trade.
*   **Mechanism:** It relies on a fixed **cost function** based on the outstanding shares. When traders buy shares of an outcome, the AMM automatically increases the price of that outcome and decreases the others. 
*   **Key Feature:** It mathematically guarantees that the market creator's worst-case financial loss is strictly bounded.

#### Market Infrastructure: AMMs vs. Limit Order Books (LOBs)
Different events require different liquidity models:
*   **AMMs (like LMSR):** 
    *   *Pros:* Perfect for niche, low-liquidity topics. They provide continuous, instant liquidity without needing to match a buyer with a seller.
    *   *Cons:* Requires the market creator to provide upfront capital, which may be lost as a subsidy to early traders.
*   **Limit Order Books (LOBs):**
    *   *Pros:* Traditional matching of buyers and sellers. Highly capital-efficient for massive, widely-followed events (e.g., presidential elections).
    *   *Cons:* Fails in low-interest markets because there isn't enough active trading to match orders.
*   **Hybrid Approach:** Platforms like **Polymarket** use LOBs for heavily speculated, high-volume events, while falling back on AMMs for niche, lower-volume markets.

---

In this lab, you will explore prediction markets both as a **participant** and as a **builder**:
- **Part 1: Using Manifold Markets** – You will create an account on a prediction market platform (Manifold Markets) and make a prediction in an actual market, observing how information might be reflected in prices.
- **Part 2: AMM Calculations** – You will solve a few short problems to solidify your understanding of how automated market makers (like LMSR and Uniswap’s constant product AMM) calculate prices and respond to trades.
- **Part 3: Coding an LMSR Market Maker** – You will examine a simplified Solidity smart contract that implements an LMSR-based prediction market for a binary outcome, with comments explaining how the mechanism works.

Let's get started!

---
---


## Part 1: Participating in a Prediction Market

- Pulse Market: https://pulse-prediction-market-frontend.vercel.app/ 
- A web application on Somnia testnet that allows users to create and participate in prediction markets. 

### How does the prediction market look like?
- Polymarket: https://polymarket.com/ 
- How does it work: https://docs.polymarket.com/ 
- A real-money prediction market platform built on Ethereum mainnet. It offers a wide range of markets on various topics, including politics, sports, and current events. Users can trade in USDC to buy shares of different outcomes, and the market prices reflect the collective belief of the participants.

‼️ Note: Polymarket operates on the Ethereum mainnet and requires real money (USDC) to participate. For this lab, we will use the Pulse Market on the Somnia testnet, which allows you to participate without risking real funds.


### Participating Instructions (for the Pulse Market)
1. Connect your wallet (e.g. MetaMask) to the web application. This will automatically switch your wallet to the Somnia testnet.
2. Check your wallet balance to ensure you have some test tokens (e.g. 0.1 STT) to participate in the market.
3. Browse the existing markets and select one that interests you. For example, you might find a market like "On April 27th, will the price of ETH be higher than 2,300 USD?".
4. You can either buy shares of "Yes" or "No" depending on your prediction. For instance, if you think ETH will be above 2,300 USD, you would buy "Yes" shares and set the amount you want to invest (e.g. 0.01 STT).
5. After placing your bet, observe how the market price changes based on your trade and any other trades that occur. The price will reflect the collective belief of all participants in the market.

----

## Part 2: AMM Calculation Exercises
Next, let's reinforce some concepts with a couple of calculation exercises. These will help you understand quantitatively how market makers set prices in prediction markets.

### Exercise 1: LMSR (Logarithmic Market Scoring Rule) Price Impact

**Background:**
LMSR is the foundational algorithm for prediction markets (used by early Polymarket and Manifold Markets). Its core concept is that the market's "total liquidity pool" is governed by a cost function $C(q_{yes}, q_{no})$, where $q_{yes}$ and $q_{no}$ are the number of shares for each outcome in the market. The cost to buy shares is simply the **difference in this cost function** before and after the trade. 

The LMSR cost function for two outcomes is 

$$C(q_{yes}, q_{no}) = b \cdot \ln\!\Big(e^{q_{yes}/b} + e^{q_{no}/b}\Big),$$

**Given:**
*   Liquidity parameter $b = 10$. Higher $b$ means more liquidity. 
*   Initial state: $q_{yes} = 0, q_{no} = 0$, which corresponds to a 50% implied probability for both outcomes.
*   After-trade state: $q_{yes} = 5, q_{no} = 0$
*   Math hints: $e^{0.5} \approx 1.6487$, $\ln 2 \approx 0.6931$

**Calculate the cost in Mana for the trader**

Cost of buying 5 "Yes" shares = Final Cost - Initial Cost = $C(5,0) - C(0,0)$

$C(0,0) = 10 \cdot \ln(e^{0/10} + e^{0/10}) = 10 \cdot \ln(1 + 1) = 10 \cdot \ln(2) \approx 6.931$ 

$C(5,0) = 10 \cdot \ln(e^{5/10} + e^{0/10}) = 10 \cdot \ln(e^{0.5} + 1) = 10 \cdot \ln(1.6487 + 1) = 10 \cdot \ln(2.6487) \approx 9.741$

$Cost = 9.741 - 6.931 = 2.81$ 

*(Conceptually: Even though you are buying 5 shares, the price dynamically increases as you buy. Therefore, the average cost per share is about 0.56 instead of 0.5, since the price moves up with each share purchased.)*

**Calculate the new implied probability of "Yes"**


The implied probability is the current "price" the system quotes for the very next infinitesimal share of "Yes".

$$p_{yes} = \frac{e^{q_{yes}/b}}{e^{q_{yes}/b} + e^{q_{no}/b}}$$

Substitute $q_{yes} = 5, q_{no} = 0, b = 10$:

$p_{yes} = \frac{e^{0.5}}{e^{0.5} + e^0} = \frac{1.6487}{1.6487 + 1} = \frac{1.6487}{2.6487} \approx 0.6225$


With a liquidity parameter of 10, buying 5 "Yes" shares costs roughly **2.81** and pushes the market's implied probability for "Yes" up from **50% to 62.25%**.


### Exercise 2: Constant-Product AMM

**Background:**
This is the most widely used model in DeFi (like Uniswap), governed by the invariant formula $X \times Y = K$ (where K is a constant). In a prediction market, the liquidity pool holds tokens representing both outcomes. **If you want to take one type of token out of the pool, you must put the other type in to maintain the curve's invariant.**

**Given:**
*   Initial token reserves: $X_{yes} = 50, X_{no} = 50$
*   The invariant (Constant $K$): $X_{yes} \times X_{no} = 2500$
*   Trade action: The trader **takes (buys)** 10 "Yes" tokens from the pool.

**Calculate how many "No" tokens the trader must put into the pool**

*   New "Yes" balance in the pool after the trade:
    $X_{yes\_new} = 50 - 10 = 40$
*   To keep $K=2500$ constant, calculate the new "No" balance required:
    $40 \times X_{no\_new} = 2500$
    $X_{no\_new} = 2500 / 40 = 62.5$
*   Tokens the trader must pay (put into the pool):
    New "No" balance - Old "No" balance = $62.5 - 50 = 12.5$ **"No" tokens**.

**Calculate the new pool reserves and new implied probability**

*   New Pool Reserves: 40 "Yes" tokens and 62.5 "No" tokens.
*   Calculate the new implied probability (price):
    In a constant-product model, a token's price is **inversely proportional** to its supply in the pool.
    The formula for the price (probability) of "Yes" is: 
    
    $P_{yes} = \frac{X_{no}}{X_{yes} + X_{no}}$ 
    
    *(Notice the numerator is the quantity of the opposing token)*
    
    $P_{yes} = \frac{62.5}{40 + 62.5} = \frac{62.5}{102.5} \approx 0.6097$ (or 60.97%)

The trader exchanged 12.5 "No" tokens for 10 "Yes" tokens. Because "Yes" tokens became scarcer in the pool, their implied probability automatically shifted from **50% to roughly 61%**.


## (OPTIONAL) Part 3: Implementing an LMSR Market Maker in Solidity

Finally, let's examine how we could implement a prediction market as a smart contract. Below is a simplified Solidity contract for a binary outcome prediction market using the LMSR mechanism. The contract acts as an automated market maker:
- It allows users to buy "Yes" or "No" outcome shares by paying Ether (which acts as the currency).
- It keeps track of the total shares and uses the LMSR formula to determine the cost of shares (and thus the price).
- When the outcome is resolved (set by the owner of the contract), holders of the winning outcome shares can redeem 1 Ether per share (losing shares become worthless).

For simplicity, this contract uses Ether as the currency and does not use an external price oracle for resolution (we assume the owner will call the resolve function with the true outcome). In a real-world scenario, you'd want a decentralized way to resolve the market (for example, using a trusted oracle or a voting mechanism).

⚠️ **Note:** Implementing the LMSR cost function involves using exponentials and logarithms, which Solidity can't do with native types directly (at least not with full precision). In this code, the cost calculation is shown conceptually. In practice, you'd use a fixed-point math library or iterative approximation to calculate the cost and price. The focus here is on the structure and logic rather than the exact math implementation. We use the ABDK libraries to do this calculation.


```javascript
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

// Import the ABDK Math library for fixed-point math functions (ln, exp, etc.)
import "abdk-libraries-solidity/ABDKMath64x64.sol";

contract LMSRPredictionMarket {
    using ABDKMath64x64 for int128;

    // Liquidity parameter for LMSR: higher = more liquidity, less price movement per trade
    uint256 public immutable b; // fixed liquidity parameter
    address public owner; // market creator who can resolve the market
    bool public marketResolved; // whether the market has been resolved
    bool public outcome; // true = Yes wins, false = No wins

    // Totals of shares purchased
    uint256 public totalYesShares; // q_Yes in the LMSR formula
    uint256 public totalNoShares; // q_No in the LMSR formula

    // Each user's Yes/No share balance
    mapping(address => uint256) public yesBalance;
    mapping(address => uint256) public noBalance;

    constructor(uint256 _b) payable {
        b = _b;
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }

    // Calculate LMSR cost for a trade using exponential scoring rule
    function calculateCost(
        uint256 qYesBefore, // total Yes shares before the trade
        uint256 qNoBefore, // total No shares before the trade
        uint256 qYesAfter, // total Yes shares after the trade
        uint256 qNoAfter // total No shares after the trade
    ) public view returns (uint256) {
        int128 b64 = ABDKMath64x64.fromUInt(b);

        // Convert quantities to fixed-point format and divide by b (q/b)
        int128 qYes1 = ABDKMath64x64.fromUInt(qYesBefore).div(b64);
        int128 qNo1 = ABDKMath64x64.fromUInt(qNoBefore).div(b64);
        int128 qYes2 = ABDKMath64x64.fromUInt(qYesAfter).div(b64);
        int128 qNo2 = ABDKMath64x64.fromUInt(qNoAfter).div(b64);

        // Compute cost before and after the trade 
        // (C(q_Yes, q_No) = b * ln(e^(q_Yes/b) + e^(q_No/b)))
        int128 cost1 = (qYes1.exp().add(qNo1.exp())).ln();
        int128 cost2 = (qYes2.exp().add(qNo2.exp())).ln();

        // Final cost = b * (cost2 - cost1)
        int128 costDelta = (cost2.sub(cost1)).mul(b64);
        return costDelta.toUInt();
    }

    // Buy Yes shares and pay Ether based on cost function
    function buyYes(uint256 amount) external payable {
        require(!marketResolved, "Market resolved");

        // Calculate cost of buying additional Yes shares
        uint256 cost = calculateCost(
            // TODO
        );

        require(msg.value >= cost, "Insufficient payment");

        // TODO: Update user balance and total shares
        

        // TODO: Refund any excess ETH sent
        
    }

    // Buy No shares and pay Ether based on cost function
    function buyNo(uint256 amount) external payable {
        require(!marketResolved, "Market resolved");

        // Calculate cost of buying additional No shares
        uint256 cost = calculateCost(
            // TODO
        );

        require(msg.value >= cost, "Insufficient payment");

        // TODO: Update user balance and total shares
        

        // TODO: Refund any excess ETH sent
    }

    // Resolve the market with the actual outcome (only callable by owner)
    function resolveMarket(bool _outcome) external onlyOwner {
        require(!marketResolved, "Already resolved");
        marketResolved = true;
        outcome = _outcome;
    }

    // Redeem winning shares for ETH (1 share = 1 ETH payout)
    function redeem() external {
        require(marketResolved, "Market not resolved");

        if (outcome) {
            // TODO: If "Yes" wins
            
        } else {
            // TODO: If "No" wins
            
        }
    }
}
```

This contract demonstrates the core logic of an LMSR-based prediction market. Notice that the **pricing mechanism** (the LMSR cost function) automatically adjusts the price based on how many shares of each side have been bought. If many "Yes" shares are purchased, the cost to buy additional Yes shares increases (and equivalently the implied probability of "Yes" goes up). The contract must hold enough Ether to pay out all winning shares; the parameter $b$ and any initial funding determine the maximum liability (worst-case payout minus collected fees).

By completing this lab, you have:
- Explored a live prediction market and seen how market probabilities reflect traders’ information.
- Practiced calculating price changes in two types of automated market makers.
- Reviewed how a smart contract can implement a prediction market mechanism.

Prediction markets continue to be an exciting area where finance, crowdsourced information, and blockchain technology intersect. We hope this lab gave you insight into how these markets function under the hood and how they can aggregate information in a decentralized way.